# Proyecto 2 – Análisis exploratorio y preprocesamiento
Dataset: *Diabetes 130-US hospitals (1999-2008)*. Este notebook documenta y justifica las
decisiones que luego se automatizan en `common/preprocessing.py` (usado por el DAG de Airflow y por la API).

In [1]:
import sys; sys.path.insert(0, '../common')
import pandas as pd, numpy as np
from preprocessing import clean_training_frame, FEATURES, TARGET, MODEL_CATEGORICAL, MODEL_NUMERIC
df = pd.read_csv('../data/diabetic_data.csv', dtype=str, keep_default_na=False)
df.columns = [c.lower().replace('-', '_') for c in df.columns]
print(df.shape); df.head(3)

(101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide_metformin,glipizide_metformin,glimepiride_pioglitazone,metformin_rosiglitazone,metformin_pioglitazone,change,diabetesmed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


## 1. Variable objetivo
`readmitted` tiene 3 clases. El problema clínico relevante es la **readmisión en menos de 30 días**, así que se define un target binario `readmitted_30 = 1 si readmitted == '<30'`.

In [2]:
df['readmitted'].value_counts(normalize=True).round(3)

readmitted
NO     0.539
>30    0.349
<30    0.112
Name: proportion, dtype: float64

## 2. Valores faltantes
En este archivo los faltantes vienen codificados como `'?'`. `'None'` en `max_glu_serum` y `A1Cresult` **no es faltante**: significa que la prueba no se realizó (categoría válida).

In [3]:
(df == '?').mean().sort_values(ascending=False).head(8).round(3)

weight               0.969
medical_specialty    0.491
payer_code           0.396
race                 0.022
diag_3               0.014
diag_2               0.004
diag_1               0.000
admission_type_id    0.000
dtype: float64

**Decisiones:**
- `weight` (97 % faltante) y `payer_code` (40 %, administrativo, no clínico) → **se eliminan**.
- `medical_specialty` (49 % faltante) → se conserva: el faltante pasa a categoría `Missing` y se dejan las 10 especialidades más frecuentes + `Other` (reduce la cardinalidad de 73 a 12).
- `race` y `diag_*` con `'?'` → categoría `Missing`.

In [4]:
print(df['medical_specialty'].nunique()); df['medical_specialty'].value_counts().head(12)

73


medical_specialty
?                             49949
InternalMedicine              14635
Emergency/Trauma               7565
Family/GeneralPractice         7440
Cardiology                     5352
Surgery-General                3099
Nephrology                     1613
Orthopedics                    1400
Orthopedics-Reconstructive     1233
Radiologist                    1140
Pulmonology                     871
Psychiatry                      854
Name: count, dtype: int64

## 3. Columnas sin información y filas a excluir

In [5]:
const = [c for c in df.columns if df[c].nunique() == 1]
print('Columnas constantes:', const)
print('Género inválido:', (df.gender == 'Unknown/Invalid').sum())
print('Altas por fallecimiento/hospicio:', df.discharge_disposition_id.isin(['11','13','14','19','20','21']).sum())

Columnas constantes: ['examide', 'citoglipton']


Género inválido: 3
Altas por fallecimiento/hospicio: 2423


- `examide` y `citoglipton` son constantes → se eliminan. Los demás medicamentos casi no varían; se conservan los más usados (`metformin`, `glipizide`, `glyburide`, `insulin`) además de `change` y `diabetesMed`.
- Se excluyen las filas con género inválido y los pacientes que **fallecieron o fueron a hospicio** (no pueden ser readmitidos).
- `encounter_id` y `patient_nbr` son identificadores, no variables predictivas (se conservan solo en la capa RAW).

## 4. Ingeniería de características
- `age` `[70-80)` → `age_num = 75` (punto medio, conserva el orden).
- `diag_1/2/3` (más de 700 códigos ICD-9) → 9 grupos clínicos.
- `total_visits = outpatient + emergency + inpatient`.
- IDs de admisión/alta/fuente se tratan como **categóricas** (son códigos, no magnitudes).

In [6]:
clean = clean_training_frame(df)
print(clean.shape)
clean[TARGET].value_counts(normalize=True).round(3)

(99340, 28)


readmitted_30
0    0.886
1    0.114
Name: proportion, dtype: float64

In [7]:
clean['diag_1_group'].value_counts()

diag_1_group
Circulatory        29680
Other              17793
Respiratory        13934
Digestive           9333
Diabetes            8661
Injury              6851
Genitourinary       5002
Musculoskeletal     4935
Neoplasms           3131
Missing               20
Name: count, dtype: int64

In [8]:
clean.groupby('number_inpatient')[TARGET].mean().head(8).round(3)  # señal fuerte: hospitalizaciones previas

number_inpatient
0    0.086
1    0.133
2    0.179
3    0.210
4    0.241
5    0.320
6    0.348
7    0.363
Name: readmitted_30, dtype: float64

## 5. Codificación y escalado (dentro del Pipeline de sklearn)
- Numéricas: imputación por mediana + `StandardScaler`.
- Categóricas: imputación `Missing` + `OneHotEncoder(handle_unknown='ignore')` → la API no falla ante categorías nuevas.
- El preprocesamiento va **dentro del modelo registrado en MLflow**, así la API aplica exactamente la misma transformación.

## 6. Partición train / validación / test
Como los datos llegan por lotes, cada registro se asigna a un subconjunto con un **hash estable** de su contenido (70/15/15). Un registro nunca cambia de subconjunto entre ejecuciones → no hay fuga del test hacia entrenamiento al reentrenar.

In [9]:
import hashlib
h = df.astype(str).agg('|'.join, axis=1).map(lambda s: int(hashlib.sha256(s.encode()).hexdigest()[:8], 16) % 100)
pd.cut(h, [-1, 69, 84, 99], labels=['train','val','test']).value_counts(normalize=True).round(3)

train    0.700
val      0.151
test     0.150
Name: proportion, dtype: float64

## 7. Métrica principal: F1 de la clase positiva
- La clase positiva es ~11 % → **accuracy no sirve**: predecir siempre "no readmite" da ~89 % de accuracy y no detecta a nadie.
- **Recall** es clínicamente importante, pero se maximiza trivialmente marcando a todos como positivos.
- **F1** equilibra recall y precisión sobre la clase de interés → se usa para **seleccionar y promover** el modelo. En MLflow también se registran recall, precision, ROC-AUC y accuracy.
- Para compensar el desbalance se usa `class_weight='balanced'`.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score, accuracy_score
X, y = clean[FEATURES], clean[TARGET]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
prep = lambda: ColumnTransformer([
    ('num', Pipeline([('i', SimpleImputer(strategy='median')), ('s', StandardScaler())]), MODEL_NUMERIC),
    ('cat', Pipeline([('i', SimpleImputer(strategy='constant', fill_value='Missing')),
                      ('o', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), MODEL_CATEGORICAL)])
models = {'baseline (siempre 0)': DummyClassifier(strategy='most_frequent'),
          'logistic_regression': LogisticRegression(C=0.5, class_weight='balanced', max_iter=1000),
          'random_forest': RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_leaf=20,
                                                  class_weight='balanced_subsample', n_jobs=-1, random_state=42)}
rows = []
for n, m in models.items():
    p = Pipeline([('prep', prep()), ('clf', m)]).fit(Xtr, ytr)
    pr = p.predict_proba(Xte)[:, 1]; yp = (pr >= .5).astype(int)
    rows.append(dict(modelo=n, accuracy=accuracy_score(yte, yp), recall=recall_score(yte, yp),
                     precision=precision_score(yte, yp, zero_division=0), f1=f1_score(yte, yp), roc_auc=roc_auc_score(yte, pr)))
pd.DataFrame(rows).round(3)

,modelo,accuracy,recall,precision,f1,roc_auc
0,baseline (siempre 0),0.886,0.000,0.000,0.000,0.500
1,logistic_regression,0.674,0.547,0.185,0.277,0.669
2,random_forest,0.686,0.538,0.190,0.281,0.674


El baseline confirma el punto: alto accuracy con F1 = 0. Los modelos con `class_weight` sacrifican accuracy pero detectan una fracción importante de las readmisiones. Este dataset es conocido por ser difícil de predecir.